# Full Reproducibility Notebook

This notebook is the reviewer-facing rerun path for the deepfake speech detection paper. It does not merely read committed result files: each experiment cell checks for its raw inputs and cached intermediates, regenerates missing cache/data by calling the original repository scripts, and records a full traceback when an external dependency is unavailable.

The notebook is intentionally explicit about long-running steps. Re-running everything from cold caches may download Hugging Face datasets, load Google Drive checkpoints, compute WavLM embeddings, and score multiple detectors. Expect GPU-backed runs to take substantially longer than the lightweight demo notebook.


## 0. Configuration and Notebook Runner

Set the toggles below before running all cells. `FORCE_REBUILD=False` means a cell reuses valid existing artifacts; set it to `True` when you want a cold regeneration. `STOP_ON_FAILURE=False` lets the notebook keep going and produce a complete failure report with tracebacks.


In [1]:
from __future__ import annotations
from pathlib import Path
import contextlib
import hashlib
import importlib
import json
import os
import shutil
import subprocess
import sys
import textwrap
import time
import traceback

ROOT = Path.cwd().resolve()
assert (ROOT / 'experiments').exists(), f'Run this notebook from the repository root, got {ROOT}'

# Reviewer toggles
FORCE_REBUILD = False          # rebuild outputs even if expected artifacts already exist
ALLOW_DOWNLOADS = True         # permit Hugging Face / Google Drive downloads if caches are missing
INSTALL_MISSING_PACKAGES = False
STOP_ON_FAILURE = False        # if False, collect tracebacks and continue
RUN_HEAVY_EXPERIMENTS = True   # if False, run preflight/bootstrap/audits only

os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('WANDB_MODE', 'disabled')
os.environ.setdefault('HF_HOME', str(ROOT / 'data' / 'huggingface'))
os.environ.setdefault('HF_DATASETS_CACHE', str(ROOT / 'data' / 'huggingface' / 'datasets'))
os.environ.setdefault('HF_HUB_CACHE', str(ROOT / 'data' / 'huggingface' / 'hub'))
os.environ.setdefault('HF_DATASETS_OFFLINE', '0' if ALLOW_DOWNLOADS else '1')
os.environ.setdefault('HF_HUB_OFFLINE', '0' if ALLOW_DOWNLOADS else '1')

secret = ROOT / 'secret.txt'
if secret.exists():
    token = secret.read_text().strip()
    if token:
        os.environ.setdefault('HF_TOKEN', token)
        os.environ.setdefault('HUGGING_FACE_HUB_TOKEN', token)

RUN_LOG = []
FAILURES = []


def rel(p: Path | str) -> str:
    p = Path(p)
    try:
        return str(p.resolve().relative_to(ROOT))
    except Exception:
        return str(p)


def sha256(path: Path, chunk=1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open('rb') as fh:
        while True:
            b = fh.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def paths_exist(paths) -> bool:
    return all(Path(p).exists() for p in paths)


def record_failure(name: str, exc_text: str):
    FAILURES.append({'step': name, 'traceback': exc_text})
    print(f'\n===== TRACEBACK FOR {name} =====')
    print(exc_text)
    print(f'===== END TRACEBACK FOR {name} =====\n')
    if STOP_ON_FAILURE:
        raise RuntimeError(f'{name} failed; see traceback above')


def run_cmd(name: str, cmd: list[str], expected_outputs=(), required_inputs=(), cwd: Path = ROOT, env_extra=None, timeout=None):
    """Run an original repo command with full stdout/stderr capture and traceback reporting."""
    expected_outputs = [Path(p) for p in expected_outputs]
    required_inputs = [Path(p) for p in required_inputs]
    start = time.time()
    print(f'\n### {name}')
    print('Command:', ' '.join(map(str, cmd)))

    missing_inputs = [p for p in required_inputs if not p.exists()]
    if missing_inputs:
        msg = 'Required inputs are missing before command execution:\n' + '\n'.join(f' - {rel(p)}' for p in missing_inputs)
        record_failure(name, msg)
        RUN_LOG.append({'name': name, 'status': 'missing_inputs', 'seconds': time.time() - start})
        return False

    if expected_outputs and paths_exist(expected_outputs) and not FORCE_REBUILD:
        print('SKIP: expected outputs already exist:')
        for p in expected_outputs:
            print(' -', rel(p))
        RUN_LOG.append({'name': name, 'status': 'skipped_existing', 'seconds': time.time() - start})
        return True

    env = os.environ.copy()
    if env_extra:
        env.update(env_extra)
    try:
        proc = subprocess.run(cmd, cwd=str(cwd), env=env, text=True, capture_output=True, timeout=timeout)
        print(proc.stdout[-12000:])
        if proc.returncode != 0:
            tb = 'Subprocess failed with exit code %s\n\nSTDOUT tail:\n%s\n\nSTDERR tail:\n%s' % (
                proc.returncode, proc.stdout[-12000:], proc.stderr[-12000:])
            record_failure(name, tb)
            RUN_LOG.append({'name': name, 'status': 'failed', 'seconds': time.time() - start})
            return False
        if proc.stderr.strip():
            print('STDERR tail:')
            print(proc.stderr[-4000:])
        missing_outputs = [p for p in expected_outputs if not p.exists()]
        if missing_outputs:
            tb = 'Command completed but expected outputs are missing:\n' + '\n'.join(f' - {rel(p)}' for p in missing_outputs)
            record_failure(name, tb)
            RUN_LOG.append({'name': name, 'status': 'missing_outputs', 'seconds': time.time() - start})
            return False
        RUN_LOG.append({'name': name, 'status': 'ok', 'seconds': time.time() - start})
        return True
    except Exception:
        record_failure(name, traceback.format_exc())
        RUN_LOG.append({'name': name, 'status': 'exception', 'seconds': time.time() - start})
        return False


def run_py(name: str, script: str, expected_outputs=(), required_inputs=(), args=(), env_extra=None, timeout=None):
    return run_cmd(name, [sys.executable, script, *map(str, args)], expected_outputs, required_inputs, ROOT, env_extra, timeout)


def import_report(modules):
    rows = []
    for m in modules:
        try:
            mod = importlib.import_module(m)
            rows.append({'module': m, 'status': 'ok', 'version': getattr(mod, '__version__', '')})
        except Exception as exc:
            rows.append({'module': m, 'status': 'missing', 'version': f'{type(exc).__name__}: {exc}'})
    return rows

print('Repo root:', ROOT)
print('ALLOW_DOWNLOADS:', ALLOW_DOWNLOADS)
print('FORCE_REBUILD:', FORCE_REBUILD)
print('RUN_HEAVY_EXPERIMENTS:', RUN_HEAVY_EXPERIMENTS)
print('HF token present:', bool(os.environ.get('HF_TOKEN')))


Repo root: /home/sagemaker-user/DeepfakeDetectionRenewed
ALLOW_DOWNLOADS: True
FORCE_REBUILD: False
RUN_HEAVY_EXPERIMENTS: True
HF token present: True


## 1. Dependency Preflight

This cell checks the Python packages used by the original scripts. It can optionally install missing packages, but by default it only reports them so the environment remains auditable.


In [2]:
import pandas as pd
required_modules = [
    'numpy', 'pandas', 'scipy', 'sklearn', 'torch', 'torchaudio', 'transformers',
    'datasets', 'huggingface_hub', 'pytorch_lightning', 'soundfile'
]
dep_df = pd.DataFrame(import_report(required_modules))
display(dep_df)
missing = dep_df[dep_df.status != 'ok']['module'].tolist()
if missing and INSTALL_MISSING_PACKAGES:
    run_cmd('install missing Python packages', [sys.executable, '-m', 'pip', 'install', *missing])
elif missing:
    print('Missing packages were not installed because INSTALL_MISSING_PACKAGES=False:', missing)


,module,status,version
0,numpy,ok,1.26.4
1,pandas,ok,2.3.3
2,scipy,ok,1.16.3
3,sklearn,ok,1.7.2
4,torch,ok,2.8.0
5,torchaudio,missing,ModuleNotFoundError: No module named 'torchaudio'
6,transformers,ok,4.57.6
7,datasets,ok,5.0.0
8,huggingface_hub,ok,0.36.0
9,pytorch_lightning,ok,2.6.5


Missing packages were not installed because INSTALL_MISSING_PACKAGES=False: ['torchaudio', 'soundfile']


## 2. Checkpoint Bootstrap

Original scripts expect checkpoints in several historical paths. This cell validates any local `models/good_models` checkpoints, downloads missing public/private Drive files when possible via `gdown`, and then creates symlinks/copies at the hard-coded paths used by the original scripts.


In [3]:
import os
import torch

GOOD = ROOT / 'models' / 'good_models'
MODELS = ROOT / 'models'
EXP_CKPTS = ROOT / 'experiments' / 'checkpoints'
MODELS.mkdir(exist_ok=True)
EXP_CKPTS.mkdir(parents=True, exist_ok=True)

DRIVE_FILES = {
    # first user-provided folder: models/good_models
    'robust_goat.ckpt': '1amWa3pEnFdf3d8Z5nU2cqTiSbApjmhob',
    'mini_goat-best-epoch=02-val-eer=0.0933.ckpt': '1p4QQNKtIT24uMyuM3tYzdVvl-uiGKns0',
    'mlaad_goat-best-epoch=05-val-eer=0.2795.ckpt': '19-S-cXCPRDmmW2dWzpolLxt35ub0KCIc',
    'mlaad_robust_goat.ckpt': '1LkLTKsnwLqQia0L7eK4vDRv8pN9IE7Bc',
    'mlaad_robust_goat_seed42-best-epoch=05-val-eer=0.3030.ckpt': '1KMIf1o37fzqDSXJuWJcVQjDDpbteiqgv',
    'mlaad_robust_goat_seed1024-best-epoch=03-val-eer=0.2976.ckpt': '16a6JPg3pZVOoBtynbZPC_iiYn80Zygaq',
    # second user-provided fallback folder, needed by ASVspoof multi-seed original scripts
    'robust_goat_seed3.ckpt': '1JgKPDeqJnUVaoOGvaOSRIrln8ZCUOEqY',
    'robust_goat_seed7.ckpt': '1wwS2Id9UYogg8LQ2DpcvgQal1LG0rtYg',
}

def download_drive(file_id: str, dest: Path):
    if dest.exists() and not FORCE_REBUILD:
        return True
    if not ALLOW_DOWNLOADS:
        record_failure(f'download {dest.name}', f'{rel(dest)} is missing and ALLOW_DOWNLOADS=False')
        return False
    try:
        import gdown
    except Exception:
        if INSTALL_MISSING_PACKAGES:
            run_cmd('install gdown', [sys.executable, '-m', 'pip', 'install', 'gdown'])
            import gdown
        else:
            record_failure(f'download {dest.name}', 'gdown is not installed. Set INSTALL_MISSING_PACKAGES=True or install gdown manually.')
            return False
    dest.parent.mkdir(parents=True, exist_ok=True)
    try:
        url = f'https://drive.google.com/uc?id={file_id}'
        print(f'Downloading {dest.name} from Drive id={file_id}')
        out = gdown.download(url, str(dest), quiet=False, fuzzy=True)
        return out is not None and dest.exists()
    except Exception:
        record_failure(f'download {dest.name}', traceback.format_exc())
        return False

# Ensure canonical good_models files exist.
for fname, fid in DRIVE_FILES.items():
    target = GOOD / fname if fname not in {'robust_goat_seed3.ckpt', 'robust_goat_seed7.ckpt'} else MODELS / fname
    if not target.exists():
        download_drive(fid, target)

# Historical path aliases used by original scripts. Prefer symlinks to avoid duplicate 500MB files.
# This is a list rather than a dict because one canonical checkpoint may need
# multiple historical names (for example mlaad_goat.ckpt and its best-epoch name).
ALIASES = [
    (GOOD / 'robust_goat.ckpt', MODELS / 'robust_goat.ckpt'),
    (GOOD / 'mini_goat-best-epoch=02-val-eer=0.0933.ckpt', MODELS / 'mini_goat.ckpt'),
    (GOOD / 'mlaad_goat-best-epoch=05-val-eer=0.2795.ckpt', EXP_CKPTS / 'mlaad_goat-best-epoch=05-val-eer=0.2795.ckpt'),
    (GOOD / 'mlaad_goat-best-epoch=05-val-eer=0.2795.ckpt', EXP_CKPTS / 'mlaad_goat.ckpt'),
    (GOOD / 'mlaad_robust_goat.ckpt', EXP_CKPTS / 'mlaad_robust_goat.ckpt'),
    (GOOD / 'mlaad_robust_goat_seed42-best-epoch=05-val-eer=0.3030.ckpt', EXP_CKPTS / 'mlaad_robust_goat_seed42-best-epoch=05-val-eer=0.3030.ckpt'),
    (GOOD / 'mlaad_robust_goat_seed1024-best-epoch=03-val-eer=0.2976.ckpt', EXP_CKPTS / 'mlaad_robust_goat_seed1024-best-epoch=03-val-eer=0.2976.ckpt'),
]

def link_or_copy(src: Path, dst: Path):
    if not src.exists():
        record_failure(f'checkpoint alias {rel(dst)}', f'Source checkpoint is missing: {rel(src)}')
        return False
    if dst.exists():
        return True
    dst.parent.mkdir(parents=True, exist_ok=True)
    try:
        os.symlink(os.path.relpath(src, dst.parent), dst)
        print('symlink', rel(dst), '->', os.readlink(dst))
    except Exception:
        shutil.copy2(src, dst)
        print('copy', rel(src), '->', rel(dst))
    return True

for src, dst in ALIASES:
    link_or_copy(src, dst)

# Validate loadability of all checkpoints that are present and planned.
torch_load_orig = torch.load
def trusted_load(*a, **k):
    k.setdefault('weights_only', False)
    return torch_load_orig(*a, **k)
torch.load = trusted_load

rows = []
for p in sorted(set(list(GOOD.glob('*.ckpt')) + list(MODELS.glob('robust_goat*.ckpt')) + list(EXP_CKPTS.glob('*.ckpt')))):
    row = {'path': rel(p), 'exists': p.exists(), 'bytes': p.stat().st_size if p.exists() else 0, 'ok': False, 'error': ''}
    if p.exists():
        try:
            obj = torch.load(p, map_location='cpu')
            sd = obj.get('state_dict', obj) if isinstance(obj, dict) else obj
            row['state_tensors'] = sum(1 for v in sd.values() if hasattr(v, 'shape')) if hasattr(sd, 'values') else 0
            row['param_count'] = sum(int(v.numel()) for v in sd.values() if hasattr(v, 'numel')) if hasattr(sd, 'values') else 0
            row['sha256_12'] = sha256(p)[:12] if not p.is_symlink() else sha256(p.resolve())[:12]
            row['ok'] = row['state_tensors'] > 0 and row['param_count'] > 0
        except Exception as exc:
            row['error'] = f'{type(exc).__name__}: {exc}'
    rows.append(row)
ckpt_df = pd.DataFrame(rows)
display(ckpt_df)
failed = ckpt_df[~ckpt_df.ok]
if len(failed):
    record_failure('checkpoint load validation', failed[['path','error']].to_string(index=False))



===== TRACEBACK FOR download robust_goat_seed3.ckpt =====
Traceback (most recent call last):
  File "/tmp/ipykernel_9404/793315007.py", line 42, in download_drive
    out = gdown.download(url, str(dest), quiet=False, fuzzy=True)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: download() got an unexpected keyword argument 'fuzzy'

===== END TRACEBACK FOR download robust_goat_seed3.ckpt =====


===== TRACEBACK FOR download robust_goat_seed7.ckpt =====
Traceback (most recent call last):
  File "/tmp/ipykernel_9404/793315007.py", line 42, in download_drive
    out = gdown.download(url, str(dest), quiet=False, fuzzy=True)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: download() got an unexpected keyword argument 'fuzzy'

===== END TRACEBACK FOR download robust_goat_seed7.ckpt =====

symlink models/good_models/robust_goat.ckpt -> good_models/robust_goat.ckpt
symlink models/good_models/mini_goat-best-epoch=02-val-eer=0.0933.ckpt -> 

,path,exists,bytes,ok,error,state_tensors,param_count,sha256_12
0,models/good_models/mlaad_goat-best-epoch=05-va...,True,530967736,True,,783,292019379,38c2ef1033c0
1,models/good_models/mlaad_goat-best-epoch=05-va...,True,530967736,True,,783,292019379,38c2ef1033c0
2,models/good_models/mlaad_robust_goat.ckpt,True,530967736,True,,783,292019379,f8869a12d59c
3,models/good_models/mlaad_robust_goat_seed1024-...,True,530967800,True,,783,292019379,50788b458564
4,models/good_models/mlaad_robust_goat_seed42-be...,True,530967800,True,,783,292019379,fb3adcd127a3
5,models/good_models/mini_goat-best-epoch=02-val...,True,530967736,True,,783,292019379,77b9bb9e2d01
6,models/good_models/mlaad_goat-best-epoch=05-va...,True,530967736,True,,783,292019379,38c2ef1033c0
7,models/good_models/mlaad_robust_goat.ckpt,True,530967736,True,,783,292019379,f8869a12d59c
8,models/good_models/mlaad_robust_goat_seed1024-...,True,530967800,True,,783,292019379,50788b458564
9,models/good_models/mlaad_robust_goat_seed42-be...,True,530967800,True,,783,292019379,fb3adcd127a3


## 3. Raw Data and Cache Bootstrap

These cells regenerate the raw/processed datasets that the experiment scripts consume. The original scripts are used wherever possible. If Hugging Face or Drive access is unavailable, the traceback is preserved in the report.


In [4]:
if ALLOW_DOWNLOADS:
    # ASVspoof cache warmup. This creates data/asvspoof_2019_la if missing.
    run_cmd(
        'warm ASVspoof2019 Hugging Face cache',
        [sys.executable, '-c', "from datasets import load_dataset; load_dataset('Bisher/as_vspoof_2019_la', cache_dir='data/asvspoof_2019_la', trust_remote_code=True); print('ASVspoof cache ready')"],
        expected_outputs=[ROOT / 'data' / 'asvspoof_2019_la'],
    )
else:
    print('ALLOW_DOWNLOADS=False; ASVspoof cache warmup skipped.')

run_py(
    'prepare MLAAD-tiny raw snapshot and processed tensors',
    'experiments/scripts/prepare_mlaad_tiny.py',
    expected_outputs=[
        ROOT / 'experiments/data/mlaad_tiny_processed/splits/train.json',
        ROOT / 'experiments/data/mlaad_tiny_processed/splits/val.json',
        ROOT / 'experiments/data/mlaad_tiny_processed/splits/test.json',
    ],
)

run_py(
    'evaluate MLAAD checkpoints to regenerate baseline_eval JSONs',
    'experiments/scripts/eval_mlaad_baseline.py',
    expected_outputs=[
        ROOT / 'experiments/results/mlaad/baseline_eval/test_in_distribution.json',
        ROOT / 'experiments/results/mlaad/baseline_eval/test_cross_language.json',
        ROOT / 'experiments/results/mlaad/baseline_eval/summary.json',
    ],
    required_inputs=[
        ROOT / 'experiments/data/mlaad_tiny_processed/splits/test.json',
        ROOT / 'experiments/checkpoints/mlaad_goat.ckpt',
        ROOT / 'experiments/checkpoints/mlaad_robust_goat.ckpt',
    ],
)

run_py(
    'prepare mini_goat ASVspoof train/val tensors',
    'experiments/results/e_mini_goat_fusion/prepare_mini_goat_data.py',
    expected_outputs=[
        ROOT / 'experiments/data/mini_goat_processed/splits/train.json',
        ROOT / 'experiments/data/mini_goat_processed/splits/val.json',
    ],
    required_inputs=[ROOT / 'data/asvspoof_2019_la'],
)



### warm ASVspoof2019 Hugging Face cache
Command: /opt/conda/bin/python -c from datasets import load_dataset; load_dataset('Bisher/as_vspoof_2019_la', cache_dir='data/asvspoof_2019_la', trust_remote_code=True); print('ASVspoof cache ready')




===== TRACEBACK FOR warm ASVspoof2019 Hugging Face cache =====
Subprocess failed with exit code 1

STDOUT tail:


STDERR tail:
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'Bisher/as_vspoof_2019_la' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/opt/conda/lib/python3.12/site-packages/datasets/load.py", line 1698, in load_dataset
    builder_instance = load_dataset_builder(
                       ^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.12/site-packages/datasets/load.py", line 1325, in load_dataset_builder
    dataset_module = dataset_module_factory(
                     ^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.12/site-packages/datasets/load.py", line 1211, in dataset_module_factory

False

## 4. Core Experiment Regeneration: Geometry, Caches, and Fusion

This is the dependency-ordered path for the main paper experiments. Each cell calls the original script. If an upstream cache is missing, the earlier cell generates it first.


In [5]:
if RUN_HEAVY_EXPERIMENTS:
    run_py(
        'I2 axis production / full MLAAD wave cache / geometry battery',
        'experiments/scripts/i2_geometry_battery.py',
        expected_outputs=[
            ROOT / 'outputs/px_wave_cache/i2_full_test_waves.npz',
            ROOT / 'experiments/results/i2_geometry_battery/utt_features.csv',
            ROOT / 'experiments/results/i2_geometry_battery/system_table.csv',
            ROOT / 'experiments/results/i2_geometry_battery/i2_stats.json',
        ],
        required_inputs=[
            ROOT / 'experiments/results/mlaad/baseline_eval/test_in_distribution.json',
            ROOT / 'experiments/checkpoints/mlaad_robust_goat.ckpt',
        ],
    )

    run_py(
        'I3 position geometry and 3-seed MLAAD logits',
        'experiments/scripts/i3_position_geometry.py',
        expected_outputs=[
            ROOT / 'experiments/results/i3_position_geometry/embeddings.npz',
            ROOT / 'experiments/results/i3_position_geometry/logits_main.npy',
            ROOT / 'experiments/results/i3_position_geometry/logits_s42.npy',
            ROOT / 'experiments/results/i3_position_geometry/logits_s1024.npy',
            ROOT / 'experiments/results/i3_position_geometry/i3_stats.json',
        ],
        required_inputs=[
            ROOT / 'outputs/px_wave_cache/i2_full_test_waves.npz',
            ROOT / 'experiments/results/i2_geometry_battery/utt_features.csv',
            ROOT / 'experiments/checkpoints/mlaad_robust_goat.ckpt',
            ROOT / 'experiments/checkpoints/mlaad_robust_goat_seed42-best-epoch=05-val-eer=0.3030.ckpt',
            ROOT / 'experiments/checkpoints/mlaad_robust_goat_seed1024-best-epoch=03-val-eer=0.2976.ckpt',
        ],
    )

    run_py(
        'I7 axis fusion',
        'experiments/scripts/i7_axis_fusion.py',
        expected_outputs=[
            ROOT / 'experiments/results/i7_axis_fusion/i7_headline.csv',
            ROOT / 'experiments/results/i7_axis_fusion/i7_stats.json',
        ],
        required_inputs=[
            ROOT / 'outputs/px_wave_cache/i2_full_test_waves.npz',
            ROOT / 'experiments/results/i3_position_geometry/embeddings.npz',
            ROOT / 'experiments/results/i3_position_geometry/logits_main.npy',
            ROOT / 'experiments/results/i3_position_geometry/logits_s42.npy',
            ROOT / 'experiments/results/i3_position_geometry/logits_s1024.npy',
        ],
    )
else:
    print('RUN_HEAVY_EXPERIMENTS=False; skipped I2/I3/I7 regeneration.')



### I2 axis production / full MLAAD wave cache / geometry battery
Command: /opt/conda/bin/python experiments/scripts/i2_geometry_battery.py


[I2] 1846 records
  waves 400/1846
  waves 800/1846
  waves 1200/1846
  waves 1600/1846


===== TRACEBACK FOR I2 axis production / full MLAAD wave cache / geometry battery =====
Subprocess failed with exit code 1

STDOUT tail:
[I2] 1846 records
  waves 400/1846
  waves 800/1846
  waves 1200/1846
  waves 1600/1846


STDERR tail:
Traceback (most recent call last):
  File "/home/sagemaker-user/DeepfakeDetectionRenewed/experiments/scripts/i2_geometry_battery.py", line 62, in <module>
    waves = np.stack(waves); ok_idx = np.array(ok_idx)
            ^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.12/site-packages/numpy/core/shape_base.py", line 445, in stack
    raise ValueError('need at least one array to stack')
ValueError: need at least one array to stack

===== END TRACEBACK FOR I2 axis production / full MLAAD wave cache / geometry battery =====


### I3 position geometry and 3-seed MLAAD logits
Command: /opt/conda/bin/python experiments/scripts/i3_position_geometry.py

===== TRACEBACK 

## 5. ASVspoof and ITW Regeneration

These scripts recreate ASVspoof causal/position artifacts and ITW transfer artifacts used by J3 and later audits.


In [6]:
if RUN_HEAVY_EXPERIMENTS:
    run_py(
        'I1 ASVspoof causal decomposition and baseline logits',
        'experiments/scripts/i1_geometry_causal_decomp.py',
        expected_outputs=[
            ROOT / 'experiments/results/i1_geometry_causal_decomp/i1_logits.npz',
            ROOT / 'experiments/results/i1_geometry_causal_decomp/i1_stats.json',
        ],
        required_inputs=[
            ROOT / 'models/robust_goat.ckpt',
            ROOT / 'models/robust_goat_seed3.ckpt',
            ROOT / 'models/robust_goat_seed7.ckpt',
        ],
    )

    run_py(
        'I4 ASVspoof position geometry',
        'experiments/scripts/i4_asvspoof_position.py',
        expected_outputs=[
            ROOT / 'experiments/results/i4_asvspoof_position/features.npz',
            ROOT / 'experiments/results/i4_asvspoof_position/i4_stats.json',
        ],
        required_inputs=[ROOT / 'experiments/results/i1_geometry_causal_decomp/i1_logits.npz'],
    )

    run_py(
        'I6 test-time geometry boost and ITW logits',
        'experiments/scripts/i6_testtime_geometry_boost.py',
        expected_outputs=[
            ROOT / 'experiments/results/i6_testtime_boost/i6_logits.npz',
            ROOT / 'experiments/results/i6_testtime_boost/i6_results.csv',
        ],
        required_inputs=[
            ROOT / 'outputs/px_wave_cache/i2_full_test_waves.npz',
            ROOT / 'experiments/checkpoints/mlaad_robust_goat.ckpt',
        ],
    )

    run_py(
        'I5 ITW transfer',
        'experiments/scripts/i5_itw_transfer.py',
        expected_outputs=[
            ROOT / 'experiments/results/i5_itw_transfer/features.npz',
            ROOT / 'experiments/results/i5_itw_transfer/utt_table.csv',
            ROOT / 'experiments/results/i5_itw_transfer/i5_stats.json',
        ],
        required_inputs=[
            ROOT / 'experiments/results/i6_testtime_boost/i6_logits.npz',
            ROOT / 'experiments/results/i3_position_geometry/embeddings.npz',
        ],
    )
else:
    print('RUN_HEAVY_EXPERIMENTS=False; skipped I1/I4/I6/I5 regeneration.')



### I1 ASVspoof causal decomposition and baseline logits
Command: /opt/conda/bin/python experiments/scripts/i1_geometry_causal_decomp.py

===== TRACEBACK FOR I1 ASVspoof causal decomposition and baseline logits =====
Required inputs are missing before command execution:
 - models/robust_goat_seed3.ckpt
 - models/robust_goat_seed7.ckpt
===== END TRACEBACK FOR I1 ASVspoof causal decomposition and baseline logits =====


### I4 ASVspoof position geometry
Command: /opt/conda/bin/python experiments/scripts/i4_asvspoof_position.py

===== TRACEBACK FOR I4 ASVspoof position geometry =====
Required inputs are missing before command execution:
 - experiments/results/i1_geometry_causal_decomp/i1_logits.npz
===== END TRACEBACK FOR I4 ASVspoof position geometry =====


### I6 test-time geometry boost and ITW logits
Command: /opt/conda/bin/python experiments/scripts/i6_testtime_geometry_boost.py

===== TRACEBACK FOR I6 test-time geometry boost and ITW logits =====
Required inputs are missing before

## 6. Paper-Level Experiments: Adaptive Axis, Prospective ASVspoof21, AASIST, mini_goat

These cells run the remaining major paper experiments through their original scripts. External AASIST assets are explicitly checked; missing assets produce a traceback and remain visible in the final report.


In [7]:
if RUN_HEAVY_EXPERIMENTS:
    run_py(
        'J3 axis-adaptive detector head',
        'experiments/scripts/j3_axis_adaptive_head.py',
        expected_outputs=[
            ROOT / 'experiments/results/j3_axis_adaptive/j3_results.csv',
            ROOT / 'experiments/results/j3_axis_adaptive/j3_summary.csv',
        ],
        required_inputs=[
            ROOT / 'outputs/px_wave_cache/i2_full_test_waves.npz',
            ROOT / 'experiments/results/i3_position_geometry/embeddings.npz',
            ROOT / 'experiments/results/i5_itw_transfer/features.npz',
            ROOT / 'experiments/results/i4_asvspoof_position/features.npz',
            ROOT / 'experiments/results/i1_geometry_causal_decomp/i1_logits.npz',
        ],
    )

    run_py(
        'J4 ASVspoof21 prospective/rotation check',
        'experiments/scripts/j4_asvspoof21_prospective.py',
        expected_outputs=[
            ROOT / 'experiments/results/j4_asvspoof21/j4_results.json',
            ROOT / 'experiments/results/j4_asvspoof21/j4_table.csv',
        ],
        required_inputs=[ROOT / 'experiments/results/i1_geometry_causal_decomp/i1_logits.npz'],
    )

    run_py(
        'J5 AASIST cross-family',
        'experiments/scripts/j5_aasist_crossfamily.py',
        expected_outputs=[ROOT / 'experiments/results/j5_aasist/j5_results.json'],
        required_inputs=[
            ROOT / 'baselines/aasist/models/AASIST.py',
            ROOT / 'baselines/aasist/models/weights/AASIST.pth',
            ROOT / 'baselines/aasist/config/AASIST.conf',
        ],
    )

    run_py(
        'J6 AASIST MLAAD fine-tuning',
        'experiments/scripts/j6_train_aasist_mlaad.py',
        expected_outputs=[ROOT / 'experiments/results/j6_aasist_mlaad/j6_results.json'],
        required_inputs=[
            ROOT / 'baselines/aasist/models/AASIST.py',
            ROOT / 'baselines/aasist/models/weights/AASIST.pth',
            ROOT / 'baselines/aasist/config/AASIST.conf',
            ROOT / 'experiments/data/mlaad_tiny_processed/splits/train.json',
        ],
    )

    run_py(
        'mini_goat score and axis fusion',
        'experiments/results/e_mini_goat_fusion/score_and_fuse_mini_goat.py',
        expected_outputs=[
            ROOT / 'experiments/results/e_mini_goat_fusion/mini_goat_fusion_results.json',
            ROOT / 'experiments/results/e_mini_goat_fusion/headline_comparison.csv',
        ],
        required_inputs=[
            ROOT / 'models/mini_goat.ckpt',
            ROOT / 'models/robust_goat.ckpt',
            ROOT / 'experiments/results/i1_geometry_causal_decomp/i1_logits.npz',
            ROOT / 'experiments/results/i4_asvspoof_position/features.npz',
        ],
    )
else:
    print('RUN_HEAVY_EXPERIMENTS=False; skipped J3/J4/J5/J6/mini_goat regeneration.')



### J3 axis-adaptive detector head
Command: /opt/conda/bin/python experiments/scripts/j3_axis_adaptive_head.py

===== TRACEBACK FOR J3 axis-adaptive detector head =====
Required inputs are missing before command execution:
 - outputs/px_wave_cache/i2_full_test_waves.npz
 - experiments/results/i3_position_geometry/embeddings.npz
 - experiments/results/i5_itw_transfer/features.npz
 - experiments/results/i4_asvspoof_position/features.npz
 - experiments/results/i1_geometry_causal_decomp/i1_logits.npz
===== END TRACEBACK FOR J3 axis-adaptive detector head =====


### J4 ASVspoof21 prospective/rotation check
Command: /opt/conda/bin/python experiments/scripts/j4_asvspoof21_prospective.py

===== TRACEBACK FOR J4 ASVspoof21 prospective/rotation check =====
Required inputs are missing before command execution:
 - experiments/results/i1_geometry_causal_decomp/i1_logits.npz
===== END TRACEBACK FOR J4 ASVspoof21 prospective/rotation check =====


### J5 AASIST cross-family
Command: /opt/conda/bin/

## 7. Audit Suite and Paper Claim Checks

Run the original audit scripts and compare regenerated metrics against paper targets. The final table deliberately flags any mismatch instead of adjusting results.


In [8]:
AUDIT_SCRIPTS = [
    'experiments/axis_audits/audit1_multiplicity.py',
    'experiments/axis_audits/audit2_sdalong_claim.py',
    'experiments/axis_audits/audit3_asvspoof_prospective.py',
    'experiments/axis_audits/audit4_axis_rotation.py',
    'experiments/axis_audits/audit5_fusion_claims.py',
    'experiments/axis_audits/audit6_itw_speaker.py',
    'experiments/axis_audits/audit7_agreement.py',
    'experiments/axis_audits/audit8_i1_causal.py',
    'experiments/axis_audits/audit9_hardness_reliability.py',
    'experiments/axis_audits/audit10_consistency.py',
]
for script in AUDIT_SCRIPTS:
    run_py(f'audit {Path(script).name}', script)



### audit audit1_multiplicity.py
Command: /opt/conda/bin/python experiments/axis_audits/audit1_multiplicity.py


total recorded tests: 83; uncorrected p<0.05: 25; global BH q<0.05: 1

headline claims after global BH:
          family       predictor     dataset  detector     rho      p  q_bh_global  q_bh_family
     i3_position        sd_along       MLAAD wavlm_gat  0.5975 0.0000       0.0000       0.0000
          i5_itw          s_orth         ITW wavlm_gat  0.5340 0.0028       0.0591       0.0128
    j6_aasist_ft        sd_along       MLAAD aasist_ft  0.3495 0.0058       0.0716       0.0176
     i4_asvspoof         s_along ASVspoof_i4 wavlm_gat -0.7143 0.0061       0.0716       0.0365
    j6_aasist_ft vel_entropy_L12       MLAAD aasist_ft  0.3417 0.0070       0.0716       0.0176
j4j5_prospective              P3  ASVspoof21 aasist_zs  0.6429 0.0204       0.1061       0.1344
j4j5_prospective              P3  ASVspoof21 wavlm_gat  0.5989 0.0336       0.1395       0.1344

i2 battery features passing their own FDR<0.05: 0

split-half winner frequency (top5):
winner
sd_along           0.767
rog_L0  



===== TRACEBACK FOR audit audit2_sdalong_claim.py =====
Subprocess failed with exit code 1

STDOUT tail:


STDERR tail:
Traceback (most recent call last):
  File "/home/sagemaker-user/DeepfakeDetectionRenewed/experiments/axis_audits/audit2_sdalong_claim.py", line 29, in <module>
    d = ac.load_mlaad()
        ^^^^^^^^^^^^^^^
  File "/home/sagemaker-user/DeepfakeDetectionRenewed/experiments/axis_audits/audit_common.py", line 37, in load_mlaad
    ok = np.load(WAVE_CACHE / "i2_full_test_waves.npz", allow_pickle=True)["ok_idx"]
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.12/site-packages/numpy/lib/npyio.py", line 427, in load
    fid = stack.enter_context(open(os_fspath(file), "rb"))
                              ^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/home/sagemaker-user/DeepfakeDetectionRenewed/outputs/px_wave_cache/i2_full_test_waves.npz'

===== END TRACEBACK FOR audit audit2_

 detector pred     rho  p_perm  p_holm_within  p_holm_all8  q_bh_all8
wavlm_gat   P1  0.2527  0.4029         0.5307       0.9783     0.8057
wavlm_gat   P2  0.1923  0.5307         0.5307       0.9783     0.8208
wavlm_gat   P3  0.5989  0.0336         0.1344       0.2353     0.1344
wavlm_gat   P4 -0.3736  0.2093         0.5307       0.9783     0.5580
aasist_zs   P1  0.1538  0.6156         0.9783       0.9783     0.8208
aasist_zs   P2  0.0659  0.8346         0.9783       0.9783     0.9539
aasist_zs   P3  0.6429  0.0204         0.0818       0.1636     0.1344
aasist_zs   P4 -0.0110  0.9783         0.9783       0.9783     0.9783

J2 prior prospective tests (all ns): 9

WavLM vs AASIST per-attack hardness agreement: rho=+0.214 (p=0.482)


===== TRACEBACK FOR audit audit3_asvspoof_prospective.py =====
Subprocess failed with exit code 1

STDOUT tail:
 detector pred     rho  p_perm  p_holm_within  p_holm_all8  q_bh_all8
wavlm_gat   P1  0.2527  0.4029         0.5307       0.9783     0.8057
wavlm_g



===== TRACEBACK FOR audit audit4_axis_rotation.py =====
Subprocess failed with exit code 1

STDOUT tail:


STDERR tail:
Traceback (most recent call last):
  File "/home/sagemaker-user/DeepfakeDetectionRenewed/experiments/axis_audits/audit4_axis_rotation.py", line 33, in <module>
    ml = ac.load_mlaad()
         ^^^^^^^^^^^^^^^
  File "/home/sagemaker-user/DeepfakeDetectionRenewed/experiments/axis_audits/audit_common.py", line 37, in load_mlaad
    ok = np.load(WAVE_CACHE / "i2_full_test_waves.npz", allow_pickle=True)["ok_idx"]
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.12/site-packages/numpy/lib/npyio.py", line 427, in load
    fid = stack.enter_context(open(os_fspath(file), "rb"))
                              ^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/home/sagemaker-user/DeepfakeDetectionRenewed/outputs/px_wave_cache/i2_full_test_waves.npz'

===== END TRACEBACK FOR audit audit

...

speakers: 49 spoof speakers in subset, 29 analyzed (>=10 utts); report says '58 speakers'
feature     rho      p  p_holm   q_bh
   ve12 -0.1424 0.4613  0.7860 0.5382
    ve9 -0.0527 0.7860  0.7860 0.7860
s_along -0.3020 0.1114  0.3341 0.1559
 s_orth  0.5340 0.0028  0.0171 0.0100
  rog12 -0.5611 0.0015  0.0108 0.0100
vmean12 -0.3837 0.0399  0.1595 0.0698
 vmean0 -0.4645 0.0111  0.0556 0.0260
s_orth: boot CI [+0.202,+0.761], jackknife range [+0.496,+0.596]
rog12: boot CI [-0.802,-0.221], jackknife range [-0.654,-0.518]
s_orth vs rog12 collinearity rho: -0.687
s_orth|rog12: [0.2108, 0.2723]  rog12|s_orth: [0.0034, 0.9858]
s_orth|n_utts: [0.5399, 0.0025]  s_orth|rms: [0.4571, 0.0127]
speaker-hardness split-half reliability: 0.716 (SB full≈0.835; predictor rho ceiling≈0.914)
done -> /home/sagemaker-user/DeepfakeDetectionRenewed/experiments/axis_audits/audits_outputs/audit6_itw_speaker


### audit audit7_agreement.py
Command: /opt/conda/bin/python experiments/axis_audits/audit7_agreement.py


n systems = 61
             aasist_ft  aasist_zs  wavlm_gat  robust_goat
aasist_ft        1.000      0.308      0.553        0.360
aasist_zs        0.308      1.000      0.320        0.544
wavlm_gat        0.553      0.320      1.000        0.301
robust_goat      0.360      0.544      0.301        1.000
                 pair                            type   rho  ci_lo  ci_hi
  aasist_ft~wavlm_gat       same-domain (MLAAD/MLAAD) 0.553  0.323  0.729
aasist_zs~robust_goat same-domain (ASVspoof/ASVspoof) 0.544  0.326  0.718
  aasist_ft~aasist_zs          same-arch cross-domain 0.308  0.045  0.538
aasist_ft~robust_goat                     cross/cross 0.360  0.077  0.599
  wavlm_gat~aasist_zs                     cross/cross 0.320  0.055  0.545
wavlm_gat~robust_goat                     cross/cross 0.301  0.037  0.537
                                       contrast  delta  ci_lo  ci_hi  p_two
  (aasist_ft~wavlm_gat) - (aasist_ft~aasist_zs)  0.242 -0.046  0.524  0.101
(aasist_ft~wavlm_gat) - (

conditions x seeds: (3, 22)
              A               B  dAUC_seedmean  p_reported  seed_consistent  p_seed_ttest  q_bh_seed
        iso_0.5        baseline        -0.0082       0.000            False        0.1966     0.2848
        iso_0.7        baseline        -0.0023       0.024            False        0.2051     0.2848
      iso_0.875        baseline        -0.0005       0.643            False        0.7281     0.7281
       iso_1.25        baseline         0.0019       0.107             True        0.1027     0.2043
      shift_0.5        baseline        -0.0029       0.031            False        0.5286     0.5540
      shift_0.7        baseline         0.0012       0.310            False        0.3668     0.4463
    shift_0.875        baseline         0.0007       0.510             True        0.0939     0.2043
     shift_1.25        baseline         0.0014       0.225            False        0.5319     0.5540
iso_0.7_ungated        baseline        -0.0124       0.000     



===== TRACEBACK FOR audit audit9_hardness_reliability.py =====
Subprocess failed with exit code 1

STDOUT tail:


STDERR tail:
Traceback (most recent call last):
  File "/home/sagemaker-user/DeepfakeDetectionRenewed/experiments/axis_audits/audit9_hardness_reliability.py", line 35, in <module>
    d = ac.load_mlaad()
        ^^^^^^^^^^^^^^^
  File "/home/sagemaker-user/DeepfakeDetectionRenewed/experiments/axis_audits/audit_common.py", line 37, in load_mlaad
    ok = np.load(WAVE_CACHE / "i2_full_test_waves.npz", allow_pickle=True)["ok_idx"]
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.12/site-packages/numpy/lib/npyio.py", line 427, in load
    fid = stack.enter_context(open(os_fspath(file), "rb"))
                              ^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/home/sagemaker-user/DeepfakeDetectionRenewed/outputs/px_wave_cache/i2_full_test_waves.npz'

===== END TRACEBACK FOR

verdict
VERIFIED        8
MISLABELED      6
MISLEADING      5
CONTRADICTED    2

[    VERIFIED] sd_along LOSO R²=0.273 on MLAAD (p=0.0005)
[  MISLABELED] table2: 'sd_along +0.316 (LOSO R²), p=0.0005'
[  MISLEADING] sd_along replicates on AASIST-FT (rho=+0.349, p=0.006)
[  MISLEADING] P3 rho=+0.599 (p=0.031) prospective, pre-registered
[    VERIFIED] P3 replicates for AASIST (rho=+0.643, p=0.018)
[  MISLEADING] 'P3 achieves 2/3 top-3 system hits'
[    VERIFIED] cos(w_MLAAD, w_ITW) = 0.05 (near-orthogonal)
[CONTRADICTED] cos(w_MLAAD, w_ASVspoof21) ≈ 0.36
[  MISLABELED] ITW-internal axis fusion: EER 0.363 -> 0.292 (WavLM-GAT)
[  MISLEADING] fusion reduces EER by up to 33 pts under shift (AASIST-ZS on ITW 0.486->0.161), 
[    VERIFIED] I7 MLAAD fusion 0.272 -> 0.163
[CONTRADICTED] iso_0.7 'small causal C effect' (dAUC=-0.0023, p=0.024)
[    VERIFIED] sub_top dAUC=-0.0320*** / sub_res +0.0039*** (direction content causal)
[  MISLABELED] 'true gated C-effect is dAUC = -0.0036'
[    VERIFIED]

In [9]:
import math
import numpy as np
import pandas as pd

checks = []
def check(name, actual, expected, tol):
    ok = actual is not None and np.isfinite(actual) and abs(float(actual) - float(expected)) <= tol
    checks.append({'metric': name, 'actual': actual, 'expected': expected, 'tol': tol, 'ok': bool(ok)})

def read_json(path):
    path = ROOT / path
    if not path.exists():
        record_failure(f'read {rel(path)}', f'Missing metric artifact: {rel(path)}')
        return {}
    return json.loads(path.read_text())

# Targets mirrored from the paper/demo audit. Keep tolerances explicit.
try:
    i7 = pd.read_csv(ROOT / 'experiments/results/i7_axis_fusion/i7_headline.csv')
    det = i7[(i7.seed == 'main') & (i7.scorer == 'detector')]['EER'].iloc[0]
    fus = i7[(i7.seed == 'main') & (i7.scorer == 'fused')]['EER'].iloc[0]
    check('I7 main-seed fused dEER', fus - det, -0.1094, 0.003)
except Exception:
    record_failure('I7 metric extraction', traceback.format_exc())

try:
    mini = read_json('experiments/results/e_mini_goat_fusion/mini_goat_fusion_results.json')
    check('mini_goat detector-alone EER', mini.get('detector_eer'), 0.14375, 0.002)
    check('mini_goat fused EER', mini.get('fused_eer'), 0.11375, 0.002)
except Exception:
    record_failure('mini_goat metric extraction', traceback.format_exc())

try:
    j3 = pd.read_csv(ROOT / 'experiments/results/j3_axis_adaptive/j3_summary.csv')
    row = j3[(j3.dataset == 'mlaad') & (j3.head == 'lda') & (j3.n_cal == 250)]
    if len(row):
        check('J3 MLAAD LDA n=250 dEER', row['dEER'].iloc[0], -0.136055, 0.010)
except Exception:
    record_failure('J3 metric extraction', traceback.format_exc())

try:
    j5 = read_json('experiments/results/j5_aasist/j5_results.json')
    check('AASIST zero-shot MLAAD baseline EER', j5.get('mlaad_baseline_eer'), 0.375959, 0.0005)
    check('AASIST zero-shot MLAAD fused EER', j5.get('mlaad_fused_eer'), 0.116490, 0.0005)
except Exception:
    record_failure('J5 metric extraction', traceback.format_exc())

try:
    j6 = read_json('experiments/results/j6_aasist_mlaad/j6_results.json')
    check('AASIST fine-tuned test EER', j6.get('test_eer'), 0.200451, 0.0005)
    check('AASIST fine-tuned sd_along rho', j6.get('sd_along_rho'), 0.349498, 0.0005)
except Exception:
    record_failure('J6 metric extraction', traceback.format_exc())

try:
    j4 = read_json('experiments/results/j4_asvspoof21/j4_results.json')
    check('ASVspoof21 WavLM P3 rho', j4.get('wavlm_p3_rho'), 0.598901, 0.0005)
    check('ASVspoof21 WavLM P3 p', j4.get('wavlm_p3_p'), 0.030554, 0.0005)
except Exception:
    record_failure('J4 metric extraction', traceback.format_exc())

metric_df = pd.DataFrame(checks)
display(metric_df)
if len(metric_df) and not metric_df['ok'].all():
    record_failure('paper metric audit', metric_df[~metric_df.ok].to_string(index=False))
elif len(metric_df):
    print('All extracted paper metric checks matched targets within tolerance.')
else:
    print('No metric checks could be extracted; see failures above.')


,metric,actual,expected,tol,ok
0,I7 main-seed fused dEER,-0.108869,-0.109400,0.0030,True
1,mini_goat detector-alone EER,NaN,0.143750,0.0020,False
2,mini_goat fused EER,NaN,0.113750,0.0020,False
3,AASIST zero-shot MLAAD baseline EER,NaN,0.375959,0.0005,False
4,AASIST zero-shot MLAAD fused EER,NaN,0.116490,0.0005,False
5,AASIST fine-tuned test EER,NaN,0.200451,0.0005,False
6,AASIST fine-tuned sd_along rho,NaN,0.349498,0.0005,False
7,ASVspoof21 WavLM P3 rho,NaN,0.598901,0.0005,False
8,ASVspoof21 WavLM P3 p,NaN,0.030554,0.0005,False



===== TRACEBACK FOR paper metric audit =====
                             metric  actual  expected    tol    ok
       mini_goat detector-alone EER     NaN  0.143750 0.0020 False
                mini_goat fused EER     NaN  0.113750 0.0020 False
AASIST zero-shot MLAAD baseline EER     NaN  0.375959 0.0005 False
   AASIST zero-shot MLAAD fused EER     NaN  0.116490 0.0005 False
         AASIST fine-tuned test EER     NaN  0.200451 0.0005 False
     AASIST fine-tuned sd_along rho     NaN  0.349498 0.0005 False
            ASVspoof21 WavLM P3 rho     NaN  0.598901 0.0005 False
              ASVspoof21 WavLM P3 p     NaN  0.030554 0.0005 False
===== END TRACEBACK FOR paper metric audit =====



## 8. Final Reproducibility Report

This cell summarizes every script run, every skip, and every traceback. A clean cold rerun should have no `failed`, `missing_inputs`, `missing_outputs`, or `exception` entries.


In [10]:
run_df = pd.DataFrame(RUN_LOG)
display(run_df)
print(f'Failures recorded: {len(FAILURES)}')
for i, failure in enumerate(FAILURES, 1):
    print('\n' + '='*100)
    print(f'FAILURE {i}: {failure["step"]}')
    print('='*100)
    print(failure['traceback'][-8000:])

report = {
    'repo_root': str(ROOT),
    'force_rebuild': FORCE_REBUILD,
    'allow_downloads': ALLOW_DOWNLOADS,
    'run_heavy_experiments': RUN_HEAVY_EXPERIMENTS,
    'run_log': RUN_LOG,
    'failures': FAILURES,
}
out = ROOT / 'experiments/results/reproducibility_notebook_report.json'
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(report, indent=2, default=str))
print('Wrote', rel(out))
if FAILURES:
    print('Reproducibility status: INCOMPLETE. See tracebacks above and report JSON.')
else:
    print('Reproducibility status: COMPLETE. All requested regeneration/audit steps succeeded.')


,name,status,seconds
0,warm ASVspoof2019 Hugging Face cache,failed,0.909501
1,prepare MLAAD-tiny raw snapshot and processed ...,failed,0.113006
2,evaluate MLAAD checkpoints to regenerate basel...,missing_inputs,0.000350
3,prepare mini_goat ASVspoof train/val tensors,missing_inputs,0.000160
4,I2 axis production / full MLAAD wave cache / g...,failed,2.175588
5,I3 position geometry and 3-seed MLAAD logits,missing_inputs,0.000481
6,I7 axis fusion,missing_inputs,0.000633
7,I1 ASVspoof causal decomposition and baseline ...,missing_inputs,0.000670
8,I4 ASVspoof position geometry,missing_inputs,0.000339
9,I6 test-time geometry boost and ITW logits,missing_inputs,0.000285


Failures recorded: 24

FAILURE 1: download robust_goat_seed3.ckpt
Traceback (most recent call last):
  File "/tmp/ipykernel_9404/793315007.py", line 42, in download_drive
    out = gdown.download(url, str(dest), quiet=False, fuzzy=True)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: download() got an unexpected keyword argument 'fuzzy'


FAILURE 2: download robust_goat_seed7.ckpt
Traceback (most recent call last):
  File "/tmp/ipykernel_9404/793315007.py", line 42, in download_drive
    out = gdown.download(url, str(dest), quiet=False, fuzzy=True)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: download() got an unexpected keyword argument 'fuzzy'


FAILURE 3: warm ASVspoof2019 Hugging Face cache
Subprocess failed with exit code 1

STDOUT tail:


STDERR tail:
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'Bisher/as_vspoof_2019_la' isn't based on a loading script and remove `trust_remot